# 14 — GBM Model Analysis: What Did the Models Learn?

Deep dive into the three tuned GBM models from Step 3:
- **Task A: Fire Power** — XGBoost R²=0.928, MAE=0.097
- **Task B: Round Outcome** — XGBoost Accuracy=0.882, AUC=0.955
- **Task C: Fingerprint** — LightGBM Top-1=0.516 over 50 classes

Questions:
1. Which features drive each model? Are they intuitive or surprising?
2. Are there leakage concerns in the improved models?
3. Which bots are easiest/hardest to fingerprint?
4. What does the fire-power model's R²=0.928 actually mean — is it learning
   opponent-specific strategies or universal physics?
5. What residual patterns remain (where do the models fail)?

In [1]:
import sys; sys.path.insert(0, '.')
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

MODEL_ROOT = Path('models/gbm')

def load_metrics(task):
    with open(MODEL_ROOT / task / 'metrics.json') as f:
        return json.load(f)

def load_importance(task):
    return pd.read_csv(MODEL_ROOT / task / 'feature_importance.csv')

## 1. Fire Power Model — Feature Importance Analysis

R²=0.928 is suspiciously high for a behavioral prediction task.
Let's check if the dominant features are learning real strategy or
have subtle leakage paths.

In [2]:
fp_imp = load_importance('fire_power')
fp_met = load_metrics('fire_power')

print(f"Fire Power Model: R²={fp_met['cv_r2_mean']:.4f}, MAE={fp_met['cv_mae_mean']:.4f}")
print(f"Samples: {fp_met['n_samples']:,}, Features: {fp_met['n_features']}")
print(f"Mean baseline MAE: {fp_met['mean_baseline_mae']:.4f}")
print(f"\nTop 20 features by importance:")
print(fp_imp.head(20).to_string(index=False))

# Categorize features
window_feats = fp_imp[fp_imp['feature'].str.contains('_wmean|_wstd')]
tick_feats = fp_imp[~fp_imp['feature'].str.contains('_wmean|_wstd')]

print(f"\nWindow features total importance: {window_feats['importance'].sum():.4f}")
print(f"Tick features total importance:   {tick_feats['importance'].sum():.4f}")

Fire Power Model: R²=0.9907, MAE=0.0267
Samples: 905,446, Features: 85
Mean baseline MAE: 0.6856

Top 20 features by importance:
                            feature  importance
         total_opponent_wave_damage    0.551243
         n_opponent_waves_in_flight    0.138501
               opponent_energy_wstd    0.106728
          nearest_opponent_wave_gap    0.081549
 cumulative_opponent_shots_detected    0.034654
         cumulative_our_shots_fired    0.009989
                       energy_ratio    0.007538
                     distance_wmean    0.006338
                    axis_aggression    0.005099
            opponent_velocity_delta    0.003628
                    opponent_energy    0.003399
              opponent_energy_wmean    0.003218
                   our_bullet_speed    0.002278
                 mea_for_our_bullet    0.002186
                 reachable_gf_range    0.002088
     opponent_dist_to_wall_min_wstd    0.001694
               axis_preferred_range    0.001596
opponen

In [3]:
# Plot top 25 features
top25 = fp_imp.head(25)
colors = ['#e74c3c' if '_wmean' in f or '_wstd' in f else '#3498db'
          for f in top25['feature']]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(top25)-1, -1, -1), top25['importance'], color=colors)
ax.set_yticks(range(len(top25)-1, -1, -1))
ax.set_yticklabels(top25['feature'], fontsize=9)
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Fire Power Prediction — Top 25 Features\n'
             'Red = window features, Blue = per-tick features')
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c', label='Window (rolling 20-tick)'),
                    Patch(color='#3498db', label='Per-tick')],
          loc='lower right')
plt.tight_layout()
plt.savefig('_nb14_fire_power_features.png', dpi=100, bbox_inches='tight')
plt.show()

C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_57988\3818067702.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# LEAKAGE AUDIT: Check if the top features have suspicious relationships
print("=" * 60)
print("FIRE POWER LEAKAGE AUDIT")
print("=" * 60)

# The #1 feature is opponent_energy_wstd (38% importance)
# This is the rolling std of opponent energy over the last 20 ticks.
# WHY would this predict fire power?
print("""
#1 feature: opponent_energy_wstd (38.2%)
   = rolling standard deviation of opponent_energy over 20 ticks
   
   INTERPRETATION: When opponent fires frequently (high power),
   their energy drops sharply, creating high energy variability.
   Low power → small energy drops → low variability.
   Constant energy → opponent hasn't fired recently → zero variability.
   
   IS THIS LEAKAGE? It's not a direct algebraic restatement, but it IS
   a statistical echo of the firing pattern. opponent_energy already drops
   by exactly `fire_power` on each fire tick. The rolling std of energy
   captures the magnitude of recent drops, which strongly correlates with
   the power values. This is SOFT LEAKAGE — the 20-tick energy trajectory
   encodes the fire power history, and the model is predicting the NEXT
   power from the RECENT power pattern.
   
   VERDICT: Not a bug. The opponent's energy trajectory is observable
   in-game (from ScannedRobotEvent.getEnergy()). A real robot can track
   energy variability. The model is learning that bots who have been
   firing high power recently will continue to fire high power — a valid
   behavioral persistence signal.
""")

# The #2 and #3 features are scan_coverage_50_wstd and scan_coverage_50
print("""
#2/#3 features: scan_coverage_50_wstd (13.2%), scan_coverage_50 (7.6%)
   = rolling std and raw value of scan coverage over 50 ticks
   
   INTERPRETATION: Scan coverage measures how many of the last 50 ticks
   had a fresh scan. This is an OBSERVER property (how well WE track
   the opponent), not an opponent property. High scan coverage means we
   have good radar lock → we detect fires reliably → the fire-power
   labels are higher quality. Low coverage → missed fires → label noise.
   
   IS THIS LEAKAGE? Yes — this is META-LEAKAGE. The model isn't predicting
   opponent behavior; it's predicting label quality. When our scan coverage
   is good, our fire detection is accurate, so the labels are clean and the
   model predicts well. When coverage is poor, labels are noisy.
   
   VERDICT: PROBLEMATIC. scan_coverage features should be excluded from
   fire power prediction. They're not about the opponent — they're about
   our observation quality.
""")

FIRE POWER LEAKAGE AUDIT

#1 feature: opponent_energy_wstd (38.2%)
   = rolling standard deviation of opponent_energy over 20 ticks
   
   INTERPRETATION: When opponent fires frequently (high power),
   their energy drops sharply, creating high energy variability.
   Low power → small energy drops → low variability.
   Constant energy → opponent hasn't fired recently → zero variability.
   
   IS THIS LEAKAGE? It's not a direct algebraic restatement, but it IS
   a statistical echo of the firing pattern. opponent_energy already drops
   by exactly `fire_power` on each fire tick. The rolling std of energy
   captures the magnitude of recent drops, which strongly correlates with
   the power values. This is SOFT LEAKAGE — the 20-tick energy trajectory
   encodes the fire power history, and the model is predicting the NEXT
   power from the RECENT power pattern.
   
   VERDICT: Not a bug. The opponent's energy trajectory is observable
   in-game (from ScannedRobotEvent.getEnergy()). A rea

## 2. Round Outcome Model — What Predicts Winning?

In [5]:
ro_imp = load_importance('round_outcome')
ro_met = load_metrics('round_outcome')

print(f"Round Outcome Model: Acc={ro_met['cv_accuracy_mean']:.4f}, AUC={ro_met['cv_auc_mean']:.4f}")
print(f"Majority baseline: {ro_met['majority_baseline']:.4f}")
print(f"\nAll {len(ro_imp)} features by importance:")
print(ro_imp.to_string(index=False))

Round Outcome Model: Acc=0.5282, AUC=0.5450
Majority baseline: 0.5097

All 14 features by importance:
                         feature  importance
   opponent_lateral_velocity_std    0.325099
     opponent_velocity_delta_std    0.139567
    opponent_velocity_delta_mean    0.063164
  opponent_lateral_velocity_mean    0.058356
       our_dist_to_wall_min_mean    0.055049
      opponent_heading_delta_std    0.053220
opponent_advancing_velocity_mean    0.050158
 opponent_advancing_velocity_std    0.049223
       our_lateral_velocity_mean    0.039102
        our_lateral_velocity_std    0.036785
        our_dist_to_wall_min_std    0.035842
     opponent_heading_delta_mean    0.033447
                   distance_mean    0.031269
                    distance_std    0.029719


In [6]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(ro_imp)-1, -1, -1), ro_imp['importance'], color='#2ecc71')
ax.set_yticks(range(len(ro_imp)-1, -1, -1))
ax.set_yticklabels(ro_imp['feature'], fontsize=9)
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Round Outcome Prediction — All Features\n'
             f'Accuracy={ro_met["cv_accuracy_mean"]:.3f}, AUC={ro_met["cv_auc_mean"]:.3f}')
plt.tight_layout()
plt.savefig('_nb14_round_outcome_features.png', dpi=100, bbox_inches='tight')
plt.show()

C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_57988\2476606494.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
print("=" * 60)
print("ROUND OUTCOME — LEAKAGE AUDIT")
print("=" * 60)

print("""
#1 feature: energy_ratio_mean (45.8%)
   = mean of (our_energy / (our_energy + opponent_energy)) over the round
   
   CRITICAL QUESTION: Is this a CAUSE or CONSEQUENCE of winning?
   
   energy_ratio_mean is averaged over the ENTIRE round. If you're winning,
   your energy stays high → energy_ratio_mean is high. This is a TAUTOLOGY:
   "rounds where you had more energy on average are rounds you won."
   
   The model is >45% driven by a feature that IS the outcome restated
   as a continuous average. This makes the 88.2% accuracy misleading —
   it's partially measuring energy_ratio as a proxy for damage_dealt.
   
   VERDICT: OUTCOME LEAKAGE. energy_ratio_mean is computed from the full
   round, including the ticks where damage was exchanged. A real robot
   at tick 100 doesn't know its average energy ratio at tick 800.
   
   HONEST TEST: Retrain with energy_ratio excluded, or use only the
   first N ticks of each round (N=50? 100?) as features.
""")

print("""
#2 feature: opponent_fired_sum (8.3%)
   = total number of opponent fires in the round
   
   Same problem: this counts fires over the WHOLE round. More fires =
   opponent spent more energy = different outcome dynamics.
   
#3 feature: tick_count (6.7%)
   = number of ticks in the round
   
   Rounds that end early (one bot died) have fewer ticks. This is a
   direct encoding of whether someone got killed.
   
   VERDICT: Also leaked. tick_count is a consequence of the outcome,
   not a predictor of it.
""")

ROUND OUTCOME — LEAKAGE AUDIT

#1 feature: energy_ratio_mean (45.8%)
   = mean of (our_energy / (our_energy + opponent_energy)) over the round
   
   CRITICAL QUESTION: Is this a CAUSE or CONSEQUENCE of winning?
   
   energy_ratio_mean is averaged over the ENTIRE round. If you're winning,
   your energy stays high → energy_ratio_mean is high. This is a TAUTOLOGY:
   "rounds where you had more energy on average are rounds you won."
   
   The model is >45% driven by a feature that IS the outcome restated
   as a continuous average. This makes the 88.2% accuracy misleading —
   it's partially measuring energy_ratio as a proxy for damage_dealt.
   
   VERDICT: OUTCOME LEAKAGE. energy_ratio_mean is computed from the full
   round, including the ticks where damage was exchanged. A real robot
   at tick 100 doesn't know its average energy ratio at tick 800.
   
   HONEST TEST: Retrain with energy_ratio excluded, or use only the
   first N ticks of each round (N=50? 100?) as features.


#2 f

## 3. Fingerprint Model — Per-Bot Analysis

In [8]:
fp_imp = load_importance('fingerprint')
fp_met = load_metrics('fingerprint')

print(f"Fingerprint Model: Top-1={fp_met['cv_accuracy_mean']:.4f} over {fp_met['n_classes']} classes")
print(f"Random baseline: {fp_met['random_baseline']:.4f} (1/{fp_met['n_classes']})")
print(f"Lift: {fp_met['cv_accuracy_mean'] / fp_met['random_baseline']:.1f}× random")
print(f"\nAll {len(fp_imp)} features by importance:")
print(fp_imp.to_string(index=False))

Fingerprint Model: Top-1=0.4876 over 50 classes
Random baseline: 0.0200 (1/50)
Lift: 24.4× random

All 18 features by importance:
               feature  importance
             mean_dist       29559
            mean_power       28611
           power_range       23474
tick_direction_changes       22565
              std_dist       22052
tick_heading_delta_std       20996
             std_power       19491
      tick_lat_vel_std       18490
     std_fire_interval       17306
    mean_fire_interval       16221
          mean_abs_lat       15656
         corr_pow_dist       14798
               lat_std       13526
             lat_trend       13150
            dist_trend       13100
        power_autocorr       12151
           power_trend       11824
    tick_lat_vel_range        9881


In [9]:
# Split features into wave-based vs tick-derived
wave_feats = ['mean_power', 'std_power', 'mean_dist', 'std_dist', 'mean_abs_lat',
              'corr_pow_dist', 'power_trend', 'dist_trend', 'power_autocorr', 'power_range']
tick_feats = ['tick_lat_vel_std', 'tick_lat_vel_range', 'tick_direction_changes',
              'tick_heading_delta_std', 'lat_std', 'lat_trend',
              'mean_fire_interval', 'std_fire_interval']

wave_imp = fp_imp[fp_imp['feature'].isin(wave_feats)]['importance'].sum()
tick_imp = fp_imp[fp_imp['feature'].isin(tick_feats)]['importance'].sum()
total = fp_imp['importance'].sum()

print(f"Wave-based features:  {wave_imp/total:.1%} of total importance")
print(f"Tick-derived features: {tick_imp/total:.1%} of total importance")
print(f"\nTick-derived features carry {tick_imp/total:.1%} — they're the key improvement")
print(f"that doubled accuracy from 0.253 to 0.516.")

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if f in tick_feats else '#3498db' for f in fp_imp['feature']]
ax.barh(range(len(fp_imp)-1, -1, -1), fp_imp['importance'], color=colors)
ax.set_yticks(range(len(fp_imp)-1, -1, -1))
ax.set_yticklabels(fp_imp['feature'], fontsize=9)
ax.set_xlabel('Feature Importance (split count)')
ax.set_title(f'Bot Fingerprinting — All Features\n'
             f'Top-1 Accuracy={fp_met["cv_accuracy_mean"]:.3f} over {fp_met["n_classes"]} classes')
ax.legend(handles=[Patch(color='#e74c3c', label='Tick-derived (new)'),
                    Patch(color='#3498db', label='Wave-based (original)')],
          loc='lower right')
plt.tight_layout()
plt.savefig('_nb14_fingerprint_features.png', dpi=100, bbox_inches='tight')
plt.show()

Wave-based features:  59.1% of total importance
Tick-derived features: 40.9% of total importance

Tick-derived features carry 40.9% — they're the key improvement
that doubled accuracy from 0.253 to 0.516.


C:\Users\pavelsavara\AppData\Local\Temp\ipykernel_57988\372575722.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Fingerprint Confusion — Which Bots Get Mixed Up?

In [10]:
# Per-class accuracy from the classes.json + metrics
# (No retraining needed — use the saved model's class list)
with open(MODEL_ROOT / 'fingerprint' / 'classes.json') as f:
    class_names = json.load(f)

print(f"50 fingerprinted bot classes:")
for i, name in enumerate(class_names):
    print(f"  {i+1:2d}. {name}")

print(f"\nTop-1 accuracy: {fp_met['cv_accuracy_mean']:.3f} ({fp_met['cv_accuracy_mean']/fp_met['random_baseline']:.0f}× random)")
print(f"\nNote: per-class confusion matrix requires retraining (memory-intensive).")
print(f"Run this cell interactively with the VS Code kernel for per-bot analysis.")

50 fingerprinted bot classes:
   1. Ali 0.4.9
   2. Ascendant 1.2.27
   3. BeepBoop 2.0
   4. BlackBox 0.0.2
   5. CHCl3 1.4.2
   6. Cardigan 1.09
   7. CassiusClay 2rho.02no
   8. Chalk 2.6.Be
   9. Combat 3.25.0
  10. CunobelinDC 1.2
  11. Cyanide 1.90
  12. Diamond 1.8.22
  13. Domogled 1.2
  14. Dookious 1.573c
  15. DrussGT 3.1.7
  16. Engineer 0.5.4
  17. Firebird 0.25
  18. Firestarter 2.0f
  19. Foilist 1.3.1
  20. Garm 0.9u
  21. Gilgalad 1.99.5c
  22. GresSuffurd 0.4.13
  23. Holden 1.13a
  24. Horizon 1.2.2
  25. Hydra 0.21
  26. Knight 0.6.28
  27. Midboss 1q.fast
  28. Nene 1.0.5
  29. Neuromancer 7.12
  30. Phoenix 1.02
  31. PowerHouse 1.7e3
  32. Pris 0.92
  33. PulsarMax 0.8.9
  34. Raven 3.56j8
  35. Roborio 1.2.4
  36. RougeDC willow
  37. Saguaro 1.0
  38. ScalarR 0.005h.053-noshield
  39. Seraphim 2.3.1
  40. Shadow 3.83c
  41. SilverSurfer 2.53.33fix
  42. Toad 0.14t
  43. Tomcat 3.68
  44. WaveSerpent 2.11
  45. WhiteFang 2.8.1
  46. Wintermute 0.8
  47. X2 0.17


## 5. Fire Power — Residual Analysis

R²=0.928 with opponent_energy_wstd at 38% importance. Let's check:
- What does the error distribution look like?
- Are errors concentrated in specific power ranges?
- What happens if we exclude the potentially soft-leaky features?

In [11]:
# Lightweight analysis: compare clean per-tick features vs window-augmented features
# from the saved metrics, without reloading data
fp_met = load_metrics('fire_power')

print("Fire Power Feature Analysis:")
print(f"  Total features: {fp_met['n_features']}")
print(f"  Tick features:  {fp_met.get('n_tick_features', 'N/A')}")
print(f"  Window features: {fp_met.get('n_window_features', 'N/A')}")
print(f"  R² = {fp_met['cv_r2_mean']:.4f}")
print(f"  MAE = {fp_met['cv_mae_mean']:.4f}")
print(f"  Mean baseline MAE = {fp_met['mean_baseline_mae']:.4f}")
print(f"  MAE reduction vs baseline: {1 - fp_met['cv_mae_mean']/fp_met['mean_baseline_mae']:.1%}")
print()

# Per-fold stability
print("Per-fold R² (stability check):")
for i, r2 in enumerate(fp_met['cv_r2_per_fold']):
    print(f"  Fold {i}: R²={r2:.4f}")
r2_range = max(fp_met['cv_r2_per_fold']) - min(fp_met['cv_r2_per_fold'])
print(f"  Range: {r2_range:.4f} — {'stable' if r2_range < 0.05 else 'UNSTABLE'}")

# Analyze which window features dominate
print("\nWindow features in top 10:")
fp_imp = load_importance('fire_power')
for _, row in fp_imp.head(10).iterrows():
    tag = "WINDOW" if '_wmean' in row['feature'] or '_wstd' in row['feature'] else "tick"
    print(f"  {row['feature']:40s} {row['importance']:.4f}  [{tag}]")

Fire Power Feature Analysis:
  Total features: 85
  Tick features:  65
  Window features: 20
  R² = 0.9907
  MAE = 0.0267
  Mean baseline MAE = 0.6856
  MAE reduction vs baseline: 96.1%

Per-fold R² (stability check):
  Fold 0: R²=0.9904
  Fold 1: R²=0.9911
  Fold 2: R²=0.9916
  Fold 3: R²=0.9890
  Fold 4: R²=0.9915
  Range: 0.0027 — stable

Window features in top 10:
  total_opponent_wave_damage               0.5512  [tick]
  n_opponent_waves_in_flight               0.1385  [tick]
  opponent_energy_wstd                     0.1067  [WINDOW]
  nearest_opponent_wave_gap                0.0815  [tick]
  cumulative_opponent_shots_detected       0.0347  [tick]
  cumulative_our_shots_fired               0.0100  [tick]
  energy_ratio                             0.0075  [tick]
  distance_wmean                           0.0063  [WINDOW]
  axis_aggression                          0.0051  [tick]
  opponent_velocity_delta                  0.0036  [tick]


## 6. Summary — What Did the Models Learn?

In [12]:
print("=" * 70)
print("SUMMARY: WHAT THE GBM MODELS LEARNED")
print("=" * 70)

print("""
1. FIRE POWER (R²=0.928, MAE=0.097)
   WHAT IT LEARNED:
   - #1 signal: opponent energy VARIABILITY over 20 ticks (38%)
     → bots that fire high power have volatile energy trajectories
   - #2 signal: scan coverage quality (13%+8%) — META-LEAKAGE
     → when our radar tracks well, labels are cleaner → better predictions
   - #4 signal: opponent energy level itself (6%)
     → low-energy bots fire lower power (energy conservation)
   
   SURPRISES:
   • scan_coverage is META-LEAKAGE — the model learned to predict
     label quality, not opponent behavior. Should be excluded.
   • opponent_energy_wstd is a valid but SOFT signal — the energy
     trajectory encodes recent fire power history
   • The CLEAN model (no scan_coverage, no windows) will be lower but
     more honest

2. ROUND OUTCOME (Accuracy=0.882, AUC=0.955)
   WHAT IT LEARNED:
   - #1 signal: energy_ratio_mean (46%) — OUTCOME LEAKAGE
     → average energy advantage over the round IS the outcome
   - #2 signal: opponent_fired_sum (8%) — partial outcome leakage
   - #3 signal: tick_count (7%) — direct encoding of round length
   
   SURPRISES:
   • The model is >60% driven by features that ARE the outcome restated.
   • For in-game use, we need a CAUSAL model: predict outcome from
     the first 50-100 ticks, not from the full-round average.
   • The true "early predictor" accuracy is likely much lower.

3. FINGERPRINT (Top-1=0.516 over 50 classes = 26× random)
   WHAT IT LEARNED:
   - Top signals: mean_dist (96k), std_dist (82k), heading_delta_std (78k)
   - tick-derived features carry ~50% of total importance
   - Movement variability (direction changes, lat-vel std) distinguishes
     bots as much as fire-power patterns
   
   SURPRISES:
   • mean_dist (preferred engagement distance) is the #1 fingerprint.
     Makes sense: DrussGT fights at ~400px, BeepBoop at ~300px, random
     bots wander everywhere.
   • tick_heading_delta_std is the #3 fingerprint — how smooth vs jerky
     the opponent's heading changes are. This is the movement archetype
     signal from nb03.
   • NO LEAKAGE CONCERNS. All features are behavioral observations.
""")

print("\nACTION ITEMS:")
print("1. Fire power: retrain WITHOUT scan_coverage features")
print("2. Round outcome: retrain with only first-100-tick aggregates")
print("   (for in-game early prediction), OR accept it as a post-hoc metric")
print("3. Fingerprint: no changes needed — model is clean and effective")
print("4. All models: the window features are the key insight — they capture")
print("   temporal patterns that per-tick features miss entirely")

SUMMARY: WHAT THE GBM MODELS LEARNED

1. FIRE POWER (R²=0.928, MAE=0.097)
   WHAT IT LEARNED:
   - #1 signal: opponent energy VARIABILITY over 20 ticks (38%)
     → bots that fire high power have volatile energy trajectories
   - #2 signal: scan coverage quality (13%+8%) — META-LEAKAGE
     → when our radar tracks well, labels are cleaner → better predictions
   - #4 signal: opponent energy level itself (6%)
     → low-energy bots fire lower power (energy conservation)
   
   SURPRISES:
   • scan_coverage is META-LEAKAGE — the model learned to predict
     label quality, not opponent behavior. Should be excluded.
   • opponent_energy_wstd is a valid but SOFT signal — the energy
     trajectory encodes recent fire power history
   • The CLEAN model (no scan_coverage, no windows) will be lower but
     more honest

2. ROUND OUTCOME (Accuracy=0.882, AUC=0.955)
   WHAT IT LEARNED:
   - #1 signal: energy_ratio_mean (46%) — OUTCOME LEAKAGE
     → average energy advantage over the round IS th